In [1]:
# Kaggle kütüphanesini yükle/güncelle
!pip install -q kaggle

# Bilgisayarına inen kaggle.json dosyasını seçip yüklemeni sağlar
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hasandemir54","key":"ae31fa66d7109da6edd816e8fd5b9cc8"}'}

In [2]:
# Kaggle klasörünü oluştur ve API anahtarını kopyala
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Veri setini indir (zip dosyası olarak iner)
!kaggle competitions download -c home-credit-default-risk

# Sadece ihtiyacın olan dosyaları zipten çıkar
# Not: Yarışmada 'application.csv' yerine 'application_train.csv' ve 'application_test.csv' bulunur.
!unzip home-credit-default-risk.zip application_train.csv application_test.csv bureau.csv bureau_balance.csv

# İstersen yer kaplamaması için zip dosyasını silebilirsin
!rm home-credit-default-risk.zip

 93% 639M/688M [00:08<00:01, 36.9MB/s]
100% 688M/688M [00:08<00:00, 81.4MB/s]
Archive:  home-credit-default-risk.zip
  inflating: application_test.csv    
  inflating: application_train.csv   
  inflating: bureau.csv              
  inflating: bureau_balance.csv      


In [3]:
import pandas as pd
import numpy as np

In [4]:
bureau = pd.read_csv('bureau.csv')
bureau_balance = pd.read_csv('bureau_balance.csv')

print("Bureau tablosu boyutu:", bureau.shape)
print("Bureau Balance tablosu boyutu:", bureau_balance.shape)

# Veriye hızlıca göz atalım
display(bureau_balance.head(3))

Bureau tablosu boyutu: (1716428, 17)
Bureau Balance tablosu boyutu: (27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C


burdan sonra yeni denemeye başlıyoruz.

In [5]:
bureau.shape

(1716428, 17)

In [6]:
bb = pd.read_csv('bureau_balance.csv')

print("Orijinal bureau_balance boyutu:", bb.shape)

# 2. STATUS sütunundaki harfleri ve rakamları matematiksel işleme uygun hale getirelim.
# 'C' (Closed/Kapalı) ve 'X' (Bilinmiyor) durumlarını risksiz kabul edip 0 yapıyoruz.
# '0' zaten gecikme yok demek, o da 0.
# '1, 2, 3, 4, 5' ise gecikme aylarını temsil ediyor, onları direkt sayıya çeviriyoruz.
status_mapping = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}

# Mapping işlemini uygulayıp yeni bir sayısal sütun yaratıyoruz
bb['STATUS_SCORE'] = bb['STATUS'].map(status_mapping)

# 3. Her bir Kredi ID'sine (SK_ID_BUREAU) göre gruplayıp sadece en kritik 3 özeti çıkarıyoruz:
bb_clean = bb.groupby('SK_ID_BUREAU').agg(
    BB_MONTH_COUNT=('MONTHS_BALANCE', 'count'),       # 1. Kredi toplam kaç ay sürmüş/kayıtlı?
    BB_MAX_OVERDUE_LEVEL=('STATUS_SCORE', 'max'),     # 2. Bu kredi tarihinde en fazla kaçıncı seviye gecikmeye düşmüş?
    BB_TOTAL_OVERDUE_SCORE=('STATUS_SCORE', 'sum')    # 3. Toplam gecikme skoru (Gecikmede geçen ayların toplam ağırlığı)
).reset_index()

print("\nTertemiz bureau_balance özet tablosu hazır! Yeni boyut:", bb_clean.shape)
display(bb_clean.head(20))

Orijinal bureau_balance boyutu: (27299925, 3)

Tertemiz bureau_balance özet tablosu hazır! Yeni boyut: (817395, 4)


,SK_ID_BUREAU,BB_MONTH_COUNT,BB_MAX_OVERDUE_LEVEL,BB_TOTAL_OVERDUE_SCORE
0,5001709,97,0,0
1,5001710,83,0,0
2,5001711,4,0,0
3,5001712,19,0,0
4,5001713,22,0,0
5,5001714,15,0,0
6,5001715,60,0,0
7,5001716,86,0,0
8,5001717,22,0,0
9,5001718,39,1,2


In [7]:
bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [8]:
(bureau.isnull().sum() / len(bureau)) * 100

,0
SK_ID_CURR,0.000000
SK_ID_BUREAU,0.000000
CREDIT_ACTIVE,0.000000
CREDIT_CURRENCY,0.000000
DAYS_CREDIT,0.000000
CREDIT_DAY_OVERDUE,0.000000
DAYS_CREDIT_ENDDATE,6.149573
DAYS_ENDDATE_FACT,36.916958
AMT_CREDIT_MAX_OVERDUE,65.513264
CNT_CREDIT_PROLONG,0.000000


In [9]:
bureau.drop(["AMT_ANNUITY","AMT_CREDIT_MAX_OVERDUE"],axis=1,inplace=True)

In [10]:
(bureau.isnull().sum() / len(bureau)) * 100

,0
SK_ID_CURR,0.000000
SK_ID_BUREAU,0.000000
CREDIT_ACTIVE,0.000000
CREDIT_CURRENCY,0.000000
DAYS_CREDIT,0.000000
CREDIT_DAY_OVERDUE,0.000000
DAYS_CREDIT_ENDDATE,6.149573
DAYS_ENDDATE_FACT,36.916958
CNT_CREDIT_PROLONG,0.000000
AMT_CREDIT_SUM,0.000757


In [11]:
bb_clean.isna().sum()

,0
SK_ID_BUREAU,0
BB_MONTH_COUNT,0
BB_MAX_OVERDUE_LEVEL,0
BB_TOTAL_OVERDUE_SCORE,0


In [12]:
# 2. SADECE Para Birimi (CREDIT_CURRENCY) sütununu siliyoruz
bureau_clean = bureau.drop(columns=['CREDIT_CURRENCY'])

# 3. Sütun isimlerini Türkçe anlamlarıyla güncelliyoruz
yeni_isimler = {
    'SK_ID_CURR': 'SK_ID_CURR (Ana Müşteri ID)',
    'SK_ID_BUREAU': 'SK_ID_BUREAU (Kredi ID)',
    'CREDIT_ACTIVE': 'CREDIT_ACTIVE (Kredi Durumu)',
    'DAYS_CREDIT': 'DAYS_CREDIT (Kredinin Çekildiği Gün)',
    'CREDIT_DAY_OVERDUE': 'CREDIT_DAY_OVERDUE (Gecikme Günü)',
    'DAYS_CREDIT_ENDDATE': 'DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü)',
    'DAYS_ENDDATE_FACT': 'DAYS_ENDDATE_FACT (Fiili Kapanış Günü)',
    'CNT_CREDIT_PROLONG': 'CNT_CREDIT_PROLONG (Vade Uzatma Sayısı)',
    'AMT_CREDIT_SUM': 'AMT_CREDIT_SUM (Toplam Kredi Limiti)',
    'AMT_CREDIT_SUM_DEBT': 'AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç)',
    'AMT_CREDIT_SUM_LIMIT': 'AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti)',
    'AMT_CREDIT_SUM_OVERDUE': 'AMT_CREDIT_SUM_OVERDUE (Gecikmiş Tutar)',
    'CREDIT_TYPE': 'CREDIT_TYPE (Kredi Türü)',
    'DAYS_CREDIT_UPDATE': 'DAYS_CREDIT_UPDATE (Son Güncelleme Günü)'
}

bureau_clean.rename(columns=yeni_isimler, inplace=True)

# 4. MODEL EĞİTİMİNE HAZIRLIK: Metinleri sayıya çevirme (One-Hot Encoding)
# Makinenin okuyabilmesi için metin içeren 'Kredi Türü' ve 'Kredi Durumu' sütunlarını dönüştürüyoruz
bureau_model_ready = pd.get_dummies(bureau_clean, columns=['CREDIT_TYPE (Kredi Türü)', 'CREDIT_ACTIVE (Kredi Durumu)'])

print("İşlem tamam! Sadece para birimi silindi, sütunlar isimlendirildi ve model için sayısallaştırıldı.\n")
display(bureau_model_ready.head())

İşlem tamam! Sadece para birimi silindi, sütunlar isimlendirildi ve model için sayısallaştırıldı.



,SK_ID_CURR (Ana Müşteri ID),SK_ID_BUREAU (Kredi ID),DAYS_CREDIT (Kredinin Çekildiği Gün),CREDIT_DAY_OVERDUE (Gecikme Günü),DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü),DAYS_ENDDATE_FACT (Fiili Kapanış Günü),CNT_CREDIT_PROLONG (Vade Uzatma Sayısı),AMT_CREDIT_SUM (Toplam Kredi Limiti),AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç),AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti),...,CREDIT_TYPE (Kredi Türü)_Loan for working capital replenishment,CREDIT_TYPE (Kredi Türü)_Microloan,CREDIT_TYPE (Kredi Türü)_Mobile operator loan,CREDIT_TYPE (Kredi Türü)_Mortgage,CREDIT_TYPE (Kredi Türü)_Real estate loan,CREDIT_TYPE (Kredi Türü)_Unknown type of loan,CREDIT_ACTIVE (Kredi Durumu)_Active,CREDIT_ACTIVE (Kredi Durumu)_Bad debt,CREDIT_ACTIVE (Kredi Durumu)_Closed,CREDIT_ACTIVE (Kredi Durumu)_Sold
0,215354,5714462,-497,0,-153.0,-153.0,0,91323.0,0.0,NaN,...,False,False,False,False,False,False,False,False,True,False
1,215354,5714463,-208,0,1075.0,NaN,0,225000.0,171342.0,NaN,...,False,False,False,False,False,False,True,False,False,False
2,215354,5714464,-203,0,528.0,NaN,0,464323.5,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
3,215354,5714465,-203,0,NaN,NaN,0,90000.0,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False
4,215354,5714466,-629,0,1197.0,NaN,0,2700000.0,NaN,NaN,...,False,False,False,False,False,False,True,False,False,False


In [13]:
bureau_clean.drop("CREDIT_DAY_OVERDUE (Gecikme Günü)",axis=1,inplace=True)

In [14]:
bureau_clean.drop("AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti)",axis=1,inplace=True)

In [15]:
bureau_merged = pd.merge(
    bureau_model_ready,
    bb_clean,
    how='left',
    left_on='SK_ID_BUREAU (Kredi ID)', # Bureau tablosunda adını böyle değiştirmiştik
    right_on='SK_ID_BUREAU'            # Balance tablosundaki orijinal adı
)
bureau_merged.drop(columns=['SK_ID_BUREAU'], inplace=True)

In [16]:
bureau_merged.head(10)

,SK_ID_CURR (Ana Müşteri ID),SK_ID_BUREAU (Kredi ID),DAYS_CREDIT (Kredinin Çekildiği Gün),CREDIT_DAY_OVERDUE (Gecikme Günü),DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü),DAYS_ENDDATE_FACT (Fiili Kapanış Günü),CNT_CREDIT_PROLONG (Vade Uzatma Sayısı),AMT_CREDIT_SUM (Toplam Kredi Limiti),AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç),AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti),...,CREDIT_TYPE (Kredi Türü)_Mortgage,CREDIT_TYPE (Kredi Türü)_Real estate loan,CREDIT_TYPE (Kredi Türü)_Unknown type of loan,CREDIT_ACTIVE (Kredi Durumu)_Active,CREDIT_ACTIVE (Kredi Durumu)_Bad debt,CREDIT_ACTIVE (Kredi Durumu)_Closed,CREDIT_ACTIVE (Kredi Durumu)_Sold,BB_MONTH_COUNT,BB_MAX_OVERDUE_LEVEL,BB_TOTAL_OVERDUE_SCORE
0,215354,5714462,-497,0,-153.0,-153.0,0,91323.00,0.00,NaN,...,False,False,False,False,False,True,False,NaN,NaN,NaN
1,215354,5714463,-208,0,1075.0,NaN,0,225000.00,171342.00,NaN,...,False,False,False,True,False,False,False,NaN,NaN,NaN
2,215354,5714464,-203,0,528.0,NaN,0,464323.50,NaN,NaN,...,False,False,False,True,False,False,False,NaN,NaN,NaN
3,215354,5714465,-203,0,NaN,NaN,0,90000.00,NaN,NaN,...,False,False,False,True,False,False,False,NaN,NaN,NaN
4,215354,5714466,-629,0,1197.0,NaN,0,2700000.00,NaN,NaN,...,False,False,False,True,False,False,False,NaN,NaN,NaN
5,215354,5714467,-273,0,27460.0,NaN,0,180000.00,71017.38,108982.62,...,False,False,False,True,False,False,False,NaN,NaN,NaN
6,215354,5714468,-43,0,79.0,NaN,0,42103.80,42103.80,0.00,...,False,False,False,True,False,False,False,NaN,NaN,NaN
7,162297,5714469,-1896,0,-1684.0,-1710.0,0,76878.45,0.00,0.00,...,False,False,False,False,False,True,False,NaN,NaN,NaN
8,162297,5714470,-1146,0,-811.0,-840.0,0,103007.70,0.00,0.00,...,False,False,False,False,False,True,False,NaN,NaN,NaN
9,162297,5714471,-1146,0,-484.0,NaN,0,4500.00,0.00,0.00,...,False,False,False,True,False,False,False,NaN,NaN,NaN


In [17]:
bureau_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 34 columns):
 #   Column                                                                 Dtype  
---  ------                                                                 -----  
 0   SK_ID_CURR (Ana Müşteri ID)                                            int64  
 1   SK_ID_BUREAU (Kredi ID)                                                int64  
 2   DAYS_CREDIT (Kredinin Çekildiği Gün)                                   int64  
 3   CREDIT_DAY_OVERDUE (Gecikme Günü)                                      int64  
 4   DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü)                             float64
 5   DAYS_ENDDATE_FACT (Fiili Kapanış Günü)                                 float64
 6   CNT_CREDIT_PROLONG (Vade Uzatma Sayısı)                                int64  
 7   AMT_CREDIT_SUM (Toplam Kredi Limiti)                                   float64
 8   AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç)   

In [18]:
# Eksik verilerin yüzdelik oranını hesaplayıp, virgülden sonra 2 basamak olacak şekilde yuvarlıyoruz
eksik_oranlari = (bureau_merged.isna().mean() * 100).round(2)

print("Sütunlardaki Boş (NaN) Veri Oranları (%):\n")
print(eksik_oranlari.sort_values(ascending=False)) # En çok boşluk olanı en üste koyar

Sütunlardaki Boş (NaN) Veri Oranları (%):

BB_MAX_OVERDUE_LEVEL                                                     54.89
BB_MONTH_COUNT                                                           54.89
BB_TOTAL_OVERDUE_SCORE                                                   54.89
DAYS_ENDDATE_FACT (Fiili Kapanış Günü)                                   36.92
AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti)                                34.48
AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç)                                  15.01
DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü)                                6.15
SK_ID_CURR (Ana Müşteri ID)                                               0.00
SK_ID_BUREAU (Kredi ID)                                                   0.00
DAYS_CREDIT (Kredinin Çekildiği Gün)                                      0.00
AMT_CREDIT_SUM_OVERDUE (Gecikmiş Tutar)                                   0.00
DAYS_CREDIT_UPDATE (Son Güncelleme Günü)                                  0.00
CREDIT_TY

burdan sonrasında yeni yapılan değişiklikler bulunuyor traine ekleme yapılabilmesi için

In [19]:
# Sütun adını arkadaşının tablosuyla kolay eşleşmesi için orijinal haline (SK_ID_CURR) getirelim
bureau_merged.rename(columns={'SK_ID_CURR (Ana Müşteri ID)': 'SK_ID_CURR'}, inplace=True)

# Her bir müşteri için geçmiş kredi verilerini özetliyoruz
# (Burada mantığına göre ortalama 'mean', toplam 'sum' veya maksimum 'max' alabilirsin)
bureau_agg = bureau_merged.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT (Kredinin Çekildiği Gün)': 'mean',
    'AMT_CREDIT_SUM (Toplam Kredi Limiti)': 'sum',
    'AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç)': 'sum',
    'AMT_CREDIT_SUM_OVERDUE (Gecikmiş Tutar)': 'sum',
    'CNT_CREDIT_PROLONG (Vade Uzatma Sayısı)': 'sum',
    'BB_MONTH_COUNT': 'mean',
    'BB_MAX_OVERDUE_LEVEL': 'max',
    'BB_TOTAL_OVERDUE_SCORE': 'sum',
    # One-hot encode ettiğin sütunların toplamını alarak "bu kişide bu kredi türünden kaç tane var" bilgisini elde ediyoruz
    'CREDIT_TYPE (Kredi Türü)_Consumer credit': 'sum',
    'CREDIT_TYPE (Kredi Türü)_Credit card': 'sum',
    'CREDIT_ACTIVE (Kredi Durumu)_Active': 'sum',
    'CREDIT_ACTIVE (Kredi Durumu)_Closed': 'sum'
}).reset_index()

In [21]:
# Sütun isimlerine "BUREAU_" ön eki ekleyelim ki arkadaşının ana tablosundaki verilerle isimleri çakışmasın
bureau_agg.columns = ['BUREAU_' + col if col != 'SK_ID_CURR' else col for col in bureau_agg.columns]

In [23]:
pd.set_option('display.max_columns', None)

In [24]:
bureau_merged.head()

,SK_ID_CURR,SK_ID_BUREAU (Kredi ID),DAYS_CREDIT (Kredinin Çekildiği Gün),CREDIT_DAY_OVERDUE (Gecikme Günü),DAYS_CREDIT_ENDDATE (Planlanan Bitiş Günü),DAYS_ENDDATE_FACT (Fiili Kapanış Günü),CNT_CREDIT_PROLONG (Vade Uzatma Sayısı),AMT_CREDIT_SUM (Toplam Kredi Limiti),AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç),AMT_CREDIT_SUM_LIMIT (Kredi Kartı Limiti),AMT_CREDIT_SUM_OVERDUE (Gecikmiş Tutar),DAYS_CREDIT_UPDATE (Son Güncelleme Günü),CREDIT_TYPE (Kredi Türü)_Another type of loan,CREDIT_TYPE (Kredi Türü)_Car loan,CREDIT_TYPE (Kredi Türü)_Cash loan (non-earmarked),CREDIT_TYPE (Kredi Türü)_Consumer credit,CREDIT_TYPE (Kredi Türü)_Credit card,CREDIT_TYPE (Kredi Türü)_Interbank credit,CREDIT_TYPE (Kredi Türü)_Loan for business development,CREDIT_TYPE (Kredi Türü)_Loan for purchase of shares (margin lending),CREDIT_TYPE (Kredi Türü)_Loan for the purchase of equipment,CREDIT_TYPE (Kredi Türü)_Loan for working capital replenishment,CREDIT_TYPE (Kredi Türü)_Microloan,CREDIT_TYPE (Kredi Türü)_Mobile operator loan,CREDIT_TYPE (Kredi Türü)_Mortgage,CREDIT_TYPE (Kredi Türü)_Real estate loan,CREDIT_TYPE (Kredi Türü)_Unknown type of loan,CREDIT_ACTIVE (Kredi Durumu)_Active,CREDIT_ACTIVE (Kredi Durumu)_Bad debt,CREDIT_ACTIVE (Kredi Durumu)_Closed,CREDIT_ACTIVE (Kredi Durumu)_Sold,BB_MONTH_COUNT,BB_MAX_OVERDUE_LEVEL,BB_TOTAL_OVERDUE_SCORE
0,215354,5714462,-497,0,-153.0,-153.0,0,91323.0,0.0,NaN,0.0,-131,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,NaN,NaN,NaN
1,215354,5714463,-208,0,1075.0,NaN,0,225000.0,171342.0,NaN,0.0,-20,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,NaN,NaN,NaN
2,215354,5714464,-203,0,528.0,NaN,0,464323.5,NaN,NaN,0.0,-16,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,NaN,NaN,NaN
3,215354,5714465,-203,0,NaN,NaN,0,90000.0,NaN,NaN,0.0,-16,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,NaN,NaN,NaN
4,215354,5714466,-629,0,1197.0,NaN,0,2700000.0,NaN,NaN,0.0,-21,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,NaN,NaN,NaN


In [26]:
bureau_merged.shape

(1716428, 34)

In [28]:
# 1. Her bir değişken için hangi istatistiği alacağımızı belirliyoruz
agregasyon_mantigi = {
    'SK_ID_BUREAU (Kredi ID)': 'count', # Kişinin toplam kaç geçmiş kredisi var?
    'AMT_CREDIT_SUM (Toplam Kredi Limiti)': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_DEBT (Güncel Kalan Borç)': ['sum', 'mean'],
    'AMT_CREDIT_SUM_OVERDUE (Gecikmiş Tutar)': ['sum', 'max'],
    'DAYS_CREDIT (Kredinin Çekildiği Gün)': ['min', 'max', 'mean'],
    'CREDIT_ACTIVE (Kredi Durumu)_Active': 'sum',
    'CREDIT_ACTIVE (Kredi Durumu)_Closed': 'sum',
    'CREDIT_TYPE (Kredi Türü)_Consumer credit': 'sum',
    'CREDIT_TYPE (Kredi Türü)_Credit card': 'sum',
    'BB_MAX_OVERDUE_LEVEL': 'max',
    'BB_TOTAL_OVERDUE_SCORE': 'sum'
}

# 2. Müşteri ID'sine göre gruplayıp yukarıdaki mantığı uyguluyoruz.
# NOT: Hata aldığın yeri 'SK_ID_CURR' olarak düzelttik.
# Eğer hala hata verirse, buraya 'SK_ID_CURR (Ana Müşteri ID)' yazmayı deneyebilirsin.
bureau_tek_satir = bureau_merged.groupby('SK_ID_CURR').agg(agregasyon_mantigi)

# 3. Sütun isimlerini tek katmanlı hale getiriyoruz
yeni_sutun_isimleri = []
for col in bureau_tek_satir.columns.values:
    # İki katmanlı ismi birleştir (Örn: 'AMT_CREDIT_SUM' ve 'max' -> 'BUREAU_AMT_CREDIT_SUM_max')
    yeni_isim = 'BUREAU_' + col[0] + '_' + col[1]
    # İsimdeki parantez ve boşlukları temizleyelim ki model sorun yaşamasın
    yeni_isim = yeni_isim.replace(' (', '_').replace(')', '').replace(' ', '_')
    yeni_sutun_isimleri.append(yeni_isim)

bureau_tek_satir.columns = yeni_sutun_isimleri

# Müşteri ID'sini index'ten tekrar normal sütun haline getiriyoruz
bureau_tek_satir = bureau_tek_satir.reset_index()

print("İşlem tamam! Yeni boyut:", bureau_tek_satir.shape)
display(bureau_tek_satir.head())

İşlem tamam! Yeni boyut: (305811, 18)


,SK_ID_CURR,BUREAU_SK_ID_BUREAU_Kredi_ID_count,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_sum,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_mean,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_max,BUREAU_AMT_CREDIT_SUM_DEBT_Güncel_Kalan_Borç_sum,BUREAU_AMT_CREDIT_SUM_DEBT_Güncel_Kalan_Borç_mean,BUREAU_AMT_CREDIT_SUM_OVERDUE_Gecikmiş_Tutar_sum,BUREAU_AMT_CREDIT_SUM_OVERDUE_Gecikmiş_Tutar_max,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_min,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_max,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_mean,BUREAU_CREDIT_ACTIVE_Kredi_Durumu_Active_sum,BUREAU_CREDIT_ACTIVE_Kredi_Durumu_Closed_sum,BUREAU_CREDIT_TYPE_Kredi_Türü_Consumer_credit_sum,BUREAU_CREDIT_TYPE_Kredi_Türü_Credit_card_sum,BUREAU_BB_MAX_OVERDUE_LEVEL_max,BUREAU_BB_TOTAL_OVERDUE_SCORE_sum
0,100001,7,1453365.000,207623.571429,378000.0,596686.5,85240.928571,0.0,0.0,-1572,-49,-735.000000,3,4,7,0,1.0,1.0
1,100002,8,865055.565,108131.945625,450000.0,245781.0,49156.200000,0.0,0.0,-1437,-103,-874.000000,2,6,4,4,1.0,27.0
2,100003,4,1017400.500,254350.125000,810000.0,0.0,0.000000,0.0,0.0,-2586,-606,-1400.750000,1,3,2,2,NaN,0.0
3,100004,2,189037.800,94518.900000,94537.8,0.0,0.000000,0.0,0.0,-1326,-408,-867.000000,0,2,2,0,NaN,0.0
4,100005,3,657126.000,219042.000000,568800.0,568408.5,189469.500000,0.0,0.0,-373,-62,-190.666667,2,1,2,1,0.0,0.0


In [29]:
bureau_tek_satir.head(10)

,SK_ID_CURR,BUREAU_SK_ID_BUREAU_Kredi_ID_count,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_sum,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_mean,BUREAU_AMT_CREDIT_SUM_Toplam_Kredi_Limiti_max,BUREAU_AMT_CREDIT_SUM_DEBT_Güncel_Kalan_Borç_sum,BUREAU_AMT_CREDIT_SUM_DEBT_Güncel_Kalan_Borç_mean,BUREAU_AMT_CREDIT_SUM_OVERDUE_Gecikmiş_Tutar_sum,BUREAU_AMT_CREDIT_SUM_OVERDUE_Gecikmiş_Tutar_max,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_min,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_max,BUREAU_DAYS_CREDIT_Kredinin_Çekildiği_Gün_mean,BUREAU_CREDIT_ACTIVE_Kredi_Durumu_Active_sum,BUREAU_CREDIT_ACTIVE_Kredi_Durumu_Closed_sum,BUREAU_CREDIT_TYPE_Kredi_Türü_Consumer_credit_sum,BUREAU_CREDIT_TYPE_Kredi_Türü_Credit_card_sum,BUREAU_BB_MAX_OVERDUE_LEVEL_max,BUREAU_BB_TOTAL_OVERDUE_SCORE_sum
0,100001,7,1453365.000,207623.571429,378000.0,596686.5,85240.928571,0.0,0.0,-1572,-49,-735.000000,3,4,7,0,1.0,1.0
1,100002,8,865055.565,108131.945625,450000.0,245781.0,49156.200000,0.0,0.0,-1437,-103,-874.000000,2,6,4,4,1.0,27.0
2,100003,4,1017400.500,254350.125000,810000.0,0.0,0.000000,0.0,0.0,-2586,-606,-1400.750000,1,3,2,2,NaN,0.0
3,100004,2,189037.800,94518.900000,94537.8,0.0,0.000000,0.0,0.0,-1326,-408,-867.000000,0,2,2,0,NaN,0.0
4,100005,3,657126.000,219042.000000,568800.0,568408.5,189469.500000,0.0,0.0,-373,-62,-190.666667,2,1,2,1,0.0,0.0
5,100007,1,146250.000,146250.000000,146250.0,0.0,0.000000,0.0,0.0,-1149,-1149,-1149.000000,0,1,1,0,NaN,0.0
6,100008,3,468445.500,156148.500000,267606.0,240057.0,80019.000000,0.0,0.0,-1097,-78,-757.333333,1,2,3,0,NaN,0.0
7,100009,18,4800811.500,266711.750000,1777500.0,1077349.5,76953.535714,0.0,0.0,-2882,-239,-1271.500000,4,14,16,2,NaN,0.0
8,100010,2,990000.000,495000.000000,675000.0,348007.5,174003.750000,0.0,0.0,-2741,-1138,-1939.500000,1,1,1,0,0.0,0.0
9,100011,4,435228.300,108807.075000,145242.0,0.0,0.000000,0.0,0.0,-2508,-1309,-1773.000000,0,4,3,1,NaN,0.0


In [30]:
bureau_tek_satir.shape

(305811, 18)

In [ ]:
# ==============================================================================
# BUREAU VERİSİNİ TEK SATIRA İNDİRGEME (AGGREGATION) MANTIĞI
# ------------------------------------------------------------------------------
# Amaç: Müşterinin (SK_ID_CURR) geçmişteki çoklu kredi kayıtlarını tek bir
# finansal profil (satır) haline getirmek. 34 kolondan, modelin kafasını
# karıştırmayacak en güçlü 18 kolonu (öz suyu) çıkarıyoruz.
#
# --- 1. KİMLİK (1 Kolon) ---
# SK_ID_CURR: Ana tablo ile birleştirmek için kullanacağımız müşteri numarası.
#
# --- 2. KREDİ SAYISI VE TÜRÜ (5 Kolon) ---
# BUREAU_SK_ID_BUREAU_count: Geçmişte çekilen toplam kredi sayısı.
# BUREAU_CREDIT_ACTIVE_Active_sum: Halen ödemesi devam eden aktif kredi sayısı.
# BUREAU_CREDIT_ACTIVE_Closed_sum: Başarıyla kapatılmış kredi sayısı.
# BUREAU_CREDIT_TYPE_Consumer_credit_sum: Toplam tüketici (ihtiyaç) kredisi sayısı.
# BUREAU_CREDIT_TYPE_Credit_card_sum: Toplam kredi kartı sayısı.
#
# --- 3. MİKTAR VE HACİM (5 Kolon) ---
# BUREAU_AMT_CREDIT_SUM_sum: Bugüne kadar alınan toplam kredi limiti.
# BUREAU_AMT_CREDIT_SUM_mean: Ortalama kredi çekme tutarı.
# BUREAU_AMT_CREDIT_SUM_max: Tek seferde çekilen en büyük kredi tutarı.
# BUREAU_AMT_CREDIT_SUM_DEBT_sum: Bankalara olan toplam güncel borç.
# BUREAU_AMT_CREDIT_SUM_DEBT_mean: Kredi başına düşen ortalama kalan borç.
#
# --- 4. GECİKME VE RİSK [EN KRİTİK BÖLÜM] (4 Kolon) ---
# BUREAU_AMT_CREDIT_SUM_OVERDUE_sum: Toplam gecikmiş (ödenmemiş) parasal tutar.
# BUREAU_AMT_CREDIT_SUM_OVERDUE_max: Tek seferde yapılan en büyük gecikme tutarı.
# BUREAU_BB_MAX_OVERDUE_LEVEL_max: (Balance'dan) Tarihindeki en kötü gecikme seviyesi.
# BUREAU_BB_TOTAL_OVERDUE_SCORE_sum: (Balance'dan) Ömür boyu yediği toplam gecikme skoru.
#
# --- 5. ZAMANLAMA (3 Kolon) ---
# BUREAU_DAYS_CREDIT_max: En son kredi çekilen tarih (Güncelliği gösterir).
# BUREAU_DAYS_CREDIT_min: İlk kredi çekilen tarih (Müşterilik kıdemi).
# BUREAU_DAYS_CREDIT_mean: Ortalama kredi çekme sıklığı.
#
# --- NEDEN 16 KOLONU SİLDİK? ---
# 1. Nadir görülen kredi türleri (Araba, Mortgage, Microloan vs.) ve durumları
#    (Bad debt, Sold) ezberi önlemek için silindi (Çoğu sıfırdı).
# 2. Çok fazla boşluk (NaN) içeren Planlanan/Fiili bitiş tarihleri silindi.
# 3. Kredi kartı limiti (%34'ü boştu) ve gereksiz vade uzatma sayıları atıldı.
# 4. Model için gürültü yaratacak tekrar eden gün/ay sayıları temizlendi.
# ==============================================================================